<a href="https://colab.research.google.com/github/ferragina/MyInformationRetrieval/blob/main/3_NER_EL_PoS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced NLP, phrases and semantics
In this lecture, we focus on Named Entity Recognition, Entity Linking, and Part of Speech Tagging.


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 45.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import requests
import json
import spacy
from spacy import displacy

## Part of Speech (PoS) Tagging


Let's now use spaCy to see the PoS tagging process, and how we can find dependencies between words in a sentence programmatically.

In [ ]:
en_model = spacy.load("en_core_web_sm")
sentence = "He was being opposed by her without any reason.\
	    A plan is being prepared by Charles for next project"
processed_sent = en_model(sentence)
for token in processed_sent:
    print(f'{token.text:{10}} {token.tag_:>{10}}\t{spacy.explain(token.tag_):<{50}}')

He                PRP	pronoun, personal                                 
was               VBD	verb, past tense                                  
being             VBG	verb, gerund or present participle                
opposed           VBN	verb, past participle                             
by                 IN	conjunction, subordinating or preposition         
her               PRP	pronoun, personal                                 
without            IN	conjunction, subordinating or preposition         
any                DT	determiner                                        
reason             NN	noun, singular or mass                            
.                   .	punctuation mark, sentence closer                 
	                 _SP	whitespace                                        
A                  DT	determiner                                        
plan               NN	noun, singular or mass                            
is                VBZ	verb, 3rd person singular pre

In [ ]:
displacy.render(processed_sent, style="dep", jupyter=True)

## Named Entity Recognition

Now we will load the spacy english model, and test its default NER pipeline on a sample text

In [ ]:
en_model = spacy.load("en_core_web_sm") # Loading the English model

text = "Apple is looking at buying U.K. startup for $1 billion"
doc = en_model(text)
displacy.render(doc, style="ent", jupyter=True)
#for entity in doc.ents:
#    print(entity.text, entity.label_)

In [ ]:
for entity in doc.ents:
    print(entity.text, " - ", entity.label_, " - ", spacy.explain(entity.label_))

Apple  -  ORG  -  Companies, agencies, institutions, etc.
U.K.  -  GPE  -  Countries, cities, states
$1 billion  -  MONEY  -  Monetary values, including unit


As we're able to see, verbs and less stopwords are ignored, while named entities are properly categorized

Let's test it on another, longer, phrase

In [ ]:
!wget https://raw.githubusercontent.com/LorenzoBellomo/InformationRetrieval/main/data/Leonardo.txt

--2025-03-20 16:06:47--  https://raw.githubusercontent.com/LorenzoBellomo/InformationRetrieval/main/data/Leonardo.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 604 [text/plain]
Saving to: ‘Leonardo.txt’

Leonardo.txt        100%[===================>]     604  --.-KB/s    in 0s      

2025-03-20 16:06:47 (18.9 MB/s) - ‘Leonardo.txt’ saved [604/604]



In [ ]:
with open("Leonardo.txt", 'r') as txt_file:
  leonardo = txt_file.read()
leonardo

'Leonardo da Vinci was an Italian Renaissance polymath whose areas of interest included invention, painting, sculpting, architecture, science, music, mathematics, engineering, literature, anatomy, geology, astronomy, botany, writing, history, and cartography. \nHe has been variously called the father of palaeontology, ichnology, and architecture, and is widely considered one of the greatest painters of all time. Leonardo is revered for his technological ingenuity. He conceptualised flying machines, a type of armoured fighting vehicle, concentrated solar power, an adding machine, and the double hull.'

In [ ]:
doc = en_model(leonardo)
displacy.render(doc, style="ent", jupyter=True)

Something goes wrong here, why is Renaissance labeled as ORG? Let's see the meaning of the various categories

In [ ]:
for entity in doc.ents:
  print(entity.text, " - ", entity.label_, " - ", spacy.explain(entity.label_))

Leonardo da Vinci  -  PERSON  -  People, including fictional
Italian  -  NORP  -  Nationalities or religious or political groups
Renaissance  -  ORG  -  Companies, agencies, institutions, etc.
one  -  CARDINAL  -  Numerals that do not fall under another type
Leonardo  -  PERSON  -  People, including fictional


This shows how crucial it is, in some situations, to train your application to your specific need.

### Finetuning NER

Now we download the Anatomical Entity Mention (AnEM) corpus from Github. It is a NER dataset on the biomedical field, stored using the CONLL format.  

In [ ]:
!wget https://raw.githubusercontent.com/juand-r/entity-recognition-datasets/master/data/AnEM/CONLL-format/data/AnEM.train
!wget https://raw.githubusercontent.com/juand-r/entity-recognition-datasets/master/data/AnEM/CONLL-format/data/AnEM.test

--2025-03-20 16:12:12--  https://raw.githubusercontent.com/juand-r/entity-recognition-datasets/master/data/AnEM/CONLL-format/data/AnEM.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1220623 (1.2M) [text/plain]
Saving to: ‘AnEM.train’

AnEM.train          100%[===================>]   1.16M  --.-KB/s    in 0.04s   

2025-03-20 16:12:12 (29.7 MB/s) - ‘AnEM.train’ saved [1220623/1220623]

--2025-03-20 16:12:12--  https://raw.githubusercontent.com/juand-r/entity-recognition-datasets/master/data/AnEM/CONLL-format/data/AnEM.test
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP req

spaCy does not accept files in the CONLL format, so we use its convert function. We need to rename files to avoid name collisions

In [ ]:
!mv AnEM.train AnEMTrain
!mv AnEM.test AnEMTest
!spacy convert AnEMTrain -c conll .
!spacy convert AnEMTest -c conll .

ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (2815 documents): AnEMTrain.spacy
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1882 documents): AnEMTest.spacy


**NOW IT IS TIME TO GENERATE THE base_config.cfg file**
INSTRUCTIONS ARE HERE: https://spacy.io/usage/training#quickstart

In [ ]:
!wget https://raw.githubusercontent.com/LorenzoBellomo/InformationRetrieval/main/base_config.cfg
!python -m spacy init fill-config base_config.cfg config.cfg

--2025-03-20 16:17:03--  https://raw.githubusercontent.com/LorenzoBellomo/InformationRetrieval/main/base_config.cfg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1713 (1.7K) [text/plain]
Saving to: ‘base_config.cfg’

base_config.cfg     100%[===================>]   1.67K  --.-KB/s    in 0s      

2025-03-20 16:17:03 (18.6 MB/s) - ‘base_config.cfg’ saved [1713/1713]

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


For reasons of lecture pacing, I limit the epochs to 6.

In [ ]:
!mkdir -p output
#!python -m spacy train config.cfg --output ./output --paths.train ./AnEMTrain.spacy --paths.dev ./AnEMTest.spacy
!python -m spacy train config.cfg --output ./output --paths.train ./AnEMTrain.spacy --paths.dev ./AnEMTest.spacy --training.max_epochs 2

ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     27.87    0.00    0.00    0.00    0.00
  0     200        308.78   1644.46    8.64   13.06    6.45    0.09
  0     400         85.48    866.23   15.94   33.76   10.43    0.16
  1     600        139.98   1002.81   25.36   33.55   20.38    0.25
  1     800        236.84   1008.64   37.06   50.41   29.30    0.37
✔ Saved pipeline to output directory
output/model-last


In [ ]:
trained_model = spacy.load("output/model-best")
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = trained_model(text)
displacy.render(doc, style="ent", jupyter=True)

/usr/local/lib/python3.11/dist-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


As we expect, a general purpose phrase yields no Named Entities

In [ ]:
text = "In the posterior (dorsal) cavity, the cranial cavity houses the brain, and the spinal cavity (or vertebral cavity) encloses the spinal cord."
doc = trained_model(text)
displacy.render(doc, style="ent", jupyter=True)

And as expected, the new biological categories appear!

Exercise:
1.   Given the query phrase, Print all the nouns.